# Sesión 2 — Entrenar y exportar

Este notebook entrena un modelo y **exporta el artefacto de despliegue**.

Lo importante de esta sesión no es el modelo: es lo que sale de aquí. Corre las celdas de
entrenamiento sin detenerte demasiado; donde vamos a pasar el rato es en la exportación.

```
artifacts/
├── pipeline.joblib     el Pipeline COMPLETO: preprocesamiento + estimador
│                       + transformación del target
├── metadata.json       el CONTRATO, en texto legible
└── example.json        un input válido y su predicción de referencia
```

---

## Antes de empezar: sube los datos

En el panel de la izquierda (el icono de carpeta 📁), **arrastra la carpeta `data`
completa** de tu proyecto.

Debe quedarte así:

```
/content/
└── data/
    └── train.csv
```

Los archivos que subas a Colab **se borran al cerrar la sesión**. Es normal, y por eso al
final vamos a descargar el artefacto en lugar de dejarlo aquí.

## 0. El entorno tiene que coincidir con el del servidor

Colab trae sus propias versiones de las librerías —y su propia versión de Python—, y **no son
necesariamente las que tu servidor tiene instaladas**.

Eso importa más de lo que parece: `joblib` no es un formato estable. Un modelo exportado con
una versión de scikit-learn y cargado con otra puede fallar al abrirse, o —peor— abrirse y
devolver números distintos sin avisar de nada.

Por eso fijamos aquí las mismas versiones que están en `backend/requirements.txt`.

Si la instalación falla, la celda te va a mostrar lo que dijo `pip` en lugar de un error de
Python. Casi siempre significa que Colab cambió de versión de Python y alguna de las
versiones fijadas ya no tiene paquete precompilado para ella.

In [ ]:
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules

# Las mismas versiones que backend/requirements.txt. Si cambias una, cambia las dos.
#
# Tienen que poder instalarse en Python 3.12 --el de tu instancia-- y en el que
# use Colab, que cambia cada tanto. Si una version no trae paquete precompilado
# para la version de Python que te toco, pip intenta compilarla y falla.
PINES = {
    "numpy": "2.1.3",
    "pandas": "2.2.3",
    "scikit-learn": "1.5.2",
    "joblib": "1.4.2",
}

print(f"Python de este entorno: {sys.version.split()[0]}")

if EN_COLAB:
    print("Instalando las versiones del proyecto (tarda un minuto)...")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"]
        + [f"{p}=={v}" for p, v in PINES.items()],
        capture_output=True,
        text=True,
    )
    if r.returncode == 0:
        print("Listo.")
    else:
        # Sin check=True: un traceback de subprocess no dice nada, y el error
        # de pip dice exactamente que paquete no se pudo instalar.
        print()
        print("La instalacion FALLO. Esto es lo que dijo pip:")
        print()
        print((r.stderr or r.stdout)[-1500:])
        print()
        print("Lo mas probable: alguna version fijada no tiene paquete")
        print("precompilado para este Python. Avisale al profesor con la")
        print(f"version que salio arriba (Python {sys.version.split()[0]}).")
else:
    print("No estas en Colab: se usan las versiones del entorno virtual del proyecto.")

### Comprueba que quedaron las versiones correctas

Si esta celda te pide reiniciar, hazlo: **Entorno de ejecución → Reiniciar sesión**, y vuelve
a correr desde aquí. Colab necesita reiniciar cuando cambia una librería que ya tenía
cargada.

In [ ]:
import importlib.metadata as md

problemas = []
for paquete, esperada in PINES.items():
    try:
        instalada = md.version(paquete)
    except md.PackageNotFoundError:
        problemas.append(f"{paquete}: no instalado")
        continue
    marca = "OK  " if instalada == esperada else "MAL "
    print(f"  {marca} {paquete:<14} {instalada}   (se espera {esperada})")
    if instalada != esperada:
        problemas.append(f"{paquete}: {instalada} en lugar de {esperada}")

if problemas:
    print()
    print("Reinicia el entorno de ejecucion y vuelve a correr desde la celda anterior:")
    print("   Entorno de ejecucion -> Reiniciar sesion")
else:
    print()
    print("El entorno coincide con el del servidor.")

## 1. Lo que necesitamos

In [ ]:
import hashlib
import json
import pathlib
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Dónde están los datos

Esta función busca `data/train.csv` hacia arriba desde donde estés. Funciona igual en Colab,
en la raíz del proyecto o en la carpeta `notebooks/`.

Si truena, es que no subiste la carpeta `data`.

In [ ]:
def encontrar_raiz():
    """Localiza la carpeta del proyecto buscando data/train.csv hacia arriba.

    Funciona en los tres sitios donde este codigo puede correr:
      · Colab, donde arrastraste la carpeta data al espacio de trabajo
      · la raiz del repositorio
      · la carpeta notebooks/

    No usa __file__ porque en Colab no existe.
    """
    try:
        candidatos = [pathlib.Path(__file__).resolve().parent.parent]
    except NameError:
        candidatos = []
    aqui = pathlib.Path.cwd()
    candidatos += [aqui, *aqui.parents]

    for base in candidatos:
        if (base / "data" / "train.csv").exists():
            return base

    raise FileNotFoundError(
        "No encuentro data/train.csv.\n\n"
        "    Si estas en Colab: arrastra la carpeta 'data' completa al panel\n"
        "    de archivos de la izquierda y vuelve a correr esta celda.\n\n"
        "    Si estas en tu maquina: corre el notebook desde la carpeta del\n"
        "    proyecto."
    )


RAIZ = encontrar_raiz()
DATOS = RAIZ / "data" / "train.csv"
ARTEFACTOS = RAIZ / "artifacts"


RAIZ = encontrar_raiz()
DATOS = RAIZ / "data" / "train.csv"
ARTEFACTOS = RAIZ / "artifacts"

print(f"proyecto : {RAIZ}")
print(f"datos    : {DATOS}")

## 3. Las diez features

No son las diez que dan el mejor RMSE. Son **las diez que un vendedor puede contestar sin
medir la casa**. Esa diferencia es una decisión de producto, y es la primera de esta sesión.

In [ ]:
MODEL_VERSION = "1.0.0"
SEMILLA = 42

# Las diez features que un vendedor real conoce. No son las diez que dan el
# mejor RMSE: son las diez que alguien puede contestar sin medir la casa.
NUMERICAS = [
    "GrLivArea",
    "OverallQual",
    "YearBuilt",
    "TotalBsmtSF",
    "GarageCars",
    "FullBath",
    "BedroomAbvGr",
    "LotArea",
]
CATEGORICAS = ["Neighborhood", "KitchenQual"]
FEATURES = NUMERICAS + CATEGORICAS
TARGET = "SalePrice"

## 4. El pipeline completo — la pieza central del módulo

Mira dónde vive cada cosa:

- El **preprocesamiento** está dentro del pipeline, no en celdas sueltas de este notebook.
- La **transformación del target** también, envuelta en `TransformedTargetRegressor`.

Por eso el servicio va a poder pasarle un `DataFrame` crudo y recibir pesos, sin saber nada
de imputaciones, codificación ni logaritmos.

La alternativa —aplicar `np.expm1()` en el código de Flask— pone conocimiento del modelo en
quien lo consume. El siguiente que use este artefacto lo va a olvidar y va a servir precios
de 12.3 dólares.

In [ ]:
def construir_pipeline():
    """El pipeline completo, en un solo objeto.

    Esta es la pieza central del modulo. El preprocesamiento NO queda fuera en
    celdas sueltas del notebook: viaja dentro del artefacto, asi que el
    servicio puede pasarle un DataFrame crudo y no necesita saber nada de
    imputaciones ni de codificacion.
    """
    preproceso = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    [
                        ("imputar", SimpleImputer(strategy="median")),
                        ("escalar", StandardScaler()),
                    ]
                ),
                NUMERICAS,
            ),
            (
                "cat",
                Pipeline(
                    [
                        ("imputar", SimpleImputer(strategy="most_frequent")),
                        (
                            "codificar",
                            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                        ),
                    ]
                ),
                CATEGORICAS,
            ),
        ]
    )

    modelo = Pipeline(
        [
            ("preproceso", preproceso),
            (
                # 100 arboles y no 300: el artefacto pesa 9 MB en lugar de 27,
                # y las metricas no empeoran. El tamaño del artefacto es una
                # decision de producto, no un detalle: tiene que viajar por git
                # hasta el servidor en cada cambio.
                "estimador",
                RandomForestRegressor(n_estimators=100, random_state=SEMILLA, n_jobs=-1),
            ),
        ]
    )

    # La transformacion del target vive DENTRO del artefacto.
    #
    # La alternativa seria aplicar np.expm1() en el codigo de Flask, y es peor:
    # pone conocimiento del modelo en quien lo consume. El siguiente que use
    # este artefacto lo va a olvidar y va a servir precios de 12.3 dolares.
    #
    # Asi, predict() devuelve pesos. Punto.
    return TransformedTargetRegressor(
        regressor=modelo, func=np.log1p, inverse_func=np.expm1
    )

## 5. Tres conjuntos, no dos

Entrenamiento para aprender, validación para decidir, prueba para reportar. **La prueba se
toca una sola vez**, al final.

In [ ]:
ARTEFACTOS.mkdir(exist_ok=True)
df = pd.read_csv(DATOS)
X = df[FEATURES]
y = df[TARGET]

# Tres conjuntos, no dos: entrenamiento para aprender, validacion para
# decidir, prueba para reportar. La prueba se toca UNA vez.
X_ent, X_resto, y_ent, y_resto = train_test_split(
    X, y, test_size=0.3, random_state=SEMILLA
)
X_val, X_prueba, y_val, y_prueba = train_test_split(
    X_resto, y_resto, test_size=0.5, random_state=SEMILLA
)

## 6. Entrenar

In [ ]:
pipeline = construir_pipeline()
pipeline.fit(X_ent, y_ent)

## 7. Medir

In [ ]:
def metricas(X_, y_):
    pred = pipeline.predict(X_)
    return {
        "rmse": round(float(np.sqrt(mean_squared_error(y_, pred))), 1),
        "mae": round(float(mean_absolute_error(y_, pred)), 1),
        "r2": round(float(r2_score(y_, pred)), 4),
    }

m_val = metricas(X_val, y_val)
m_prueba = metricas(X_prueba, y_prueba)

In [ ]:
print("validacion:", m_val)
print("prueba    :", m_prueba)

## 8. Comprobación antes de exportar

`predict()` **tiene que devolver pesos**, no logaritmos. Si aquí sale un número entre 10 y 14,
la transformación del target quedó fuera del artefacto y el servicio va a servir basura.

In [ ]:
prediccion = pipeline.predict(X_prueba.iloc[[0]])[0]
print(f"prediccion: {prediccion:,.2f}")
assert 10_000 < prediccion < 1_000_000, "esto no son pesos: revisa el TransformedTargetRegressor"
print("OK: predict() devuelve unidades del dominio")

---

# Aquí empieza lo que de verdad importa

---

## 9. Exportar el pipeline

Un solo archivo, con todo dentro.

In [ ]:
# --- pipeline.joblib ---
ruta_pipeline = ARTEFACTOS / "pipeline.joblib"
joblib.dump(pipeline, ruta_pipeline)
hash_artefacto = hashlib.sha256(ruta_pipeline.read_bytes()).hexdigest()[:12]

## 10. Las importancias, traducidas a las features originales

El modelo ve las columnas **después** del one-hot: `Neighborhood_NAmes`,
`Neighborhood_CollgCr`... Para el contrato queremos la importancia de `Neighborhood`
completo, así que sumamos las columnas que salieron de él.

Esto se hace **aquí y no en el servicio**: el servicio no debería tener que hurgar dentro de
un pipeline anidado para saber qué feature pesa más.

In [ ]:
def importancias_por_feature(ttr):
    """Traduce las importancias del modelo a las features originales.

    El modelo ve las columnas DESPUES del one-hot: 'Neighborhood_NAmes',
    'Neighborhood_CollgCr'... Para el contrato queremos la importancia de
    'Neighborhood' completo, asi que sumamos las columnas que salieron de el.
    """
    estimador = ttr.regressor_.named_steps["estimador"]
    nombres = ttr.regressor_.named_steps["preproceso"].get_feature_names_out()
    pesos = estimador.feature_importances_

    acumulado = {f: 0.0 for f in FEATURES}
    for nombre, peso in zip(nombres, pesos):
        sin_prefijo = nombre.split("__", 1)[1]
        for f in FEATURES:
            if sin_prefijo == f or sin_prefijo.startswith(f + "_"):
                acumulado[f] += float(peso)
                break
    return dict(sorted(acumulado.items(), key=lambda kv: kv[1], reverse=True))

## 11. `metadata.json` — el contrato

Un archivo, tres consumidores:

```
                    ┌──▶ el servicio VALIDA la entrada contra él
   metadata.json ───┼──▶ la Model Card se RENDERIZA de él
                    └──▶ la rúbrica de tu reto se EVIDENCIA con él
```

Fíjate en `sklearn_version`. Es la razón por la que la primera celda fijó las versiones: ese
campo va a viajar con el artefacto, y el servicio lo va a comparar contra lo que tenga
instalado al arrancar.

In [ ]:
# --- metadata.json: el contrato ---
features_contrato = []
for f in NUMERICAS:
    features_contrato.append(
        {
            "name": f,
            "type": "num",
            "min": float(df[f].min()),
            "max": float(df[f].max()),
            "median": float(df[f].median()),
        }
    )
for f in CATEGORICAS:
    features_contrato.append(
        {
            "name": f,
            "type": "cat",
            "allowed": sorted(df[f].dropna().unique().tolist()),
        }
    )

metadata = {
    "model_version": MODEL_VERSION,
    "trained_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sklearn_version": sklearn.__version__,
    "artifact_hash": hash_artefacto,
    "algorithm": "RandomForestRegressor(n_estimators=100)",
    "target": TARGET,
    "target_transform": "log1p",
    "features": features_contrato,
    "splits": {
        "train": int(len(X_ent)),
        "validation": int(len(X_val)),
        "test": int(len(X_prueba)),
    },
    "metrics": {"validation": m_val, "test": m_prueba},
    "feature_importances": {
        k: round(v, 4) for k, v in importancias_por_feature(pipeline).items()
    },
    # Campos que este modulo deja vacios y que cada equipo llena en su reto.
    # Las vistas del tablero ya los saben leer: aparecen como "sin datos".
    "model_comparison": [],
    "hyperparameter_experiments": [],
}
(ARTEFACTOS / "metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False) + "\n"
)

## 12. `example.json` — el smoke test

Un input válido y la predicción que este modelo le da **ahora mismo**. Es lo que va a
permitir comprobar que el servicio devuelve el mismo número que este notebook.

Ese es `tests/test_paridad.py`, y es el único test que de verdad importa en este módulo.

In [ ]:
# --- example.json: el smoke test ---
fila = X_prueba.iloc[0]
ejemplo = {
    "input": {
        k: (int(v) if isinstance(v, (np.integer,)) else
            float(v) if isinstance(v, (np.floating,)) else v)
        for k, v in fila.to_dict().items()
    },
    "prediction": round(float(pipeline.predict(X_prueba.iloc[[0]])[0]), 2),
    "model_version": MODEL_VERSION,
}
(ARTEFACTOS / "example.json").write_text(
    json.dumps(ejemplo, indent=2, ensure_ascii=False) + "\n"
)

## 13. Resumen

In [ ]:
print("Artefacto exportado en artifacts/")
print(f"  splits          : {metadata['splits']}")
print(f"  RMSE validacion : {m_val['rmse']:,.0f}")
print(f"  RMSE prueba     : {m_prueba['rmse']:,.0f}")
print(f"  R2 prueba       : {m_prueba['r2']}")
print(f"  sklearn         : {metadata['sklearn_version']}")
print(f"  hash            : {hash_artefacto}")
print(f"  prediccion ref  : {ejemplo['prediction']:,.2f}")
print()
print("  importancias:")
for k, v in list(metadata["feature_importances"].items())[:5]:
    print(f"    {k:<16} {v}")

## 14. Descarga el artefacto

Colab borra tus archivos al cerrar la sesión, así que el artefacto tiene que salir de aquí.

Esta celda te descarga un `.zip` con los tres archivos. **Descomprímelo en tu proyecto**, de
forma que quede en `artifacts/`, y súbelo con el resto de tu código:

```bash
git add artifacts/
git commit -m "artefacto de la sesion 2"
git push
```

Y en tu instancia: `./setup/run sync && ./setup/run restart`.

Fíjate en lo que acaba de pasar: **entrenaste en un lado y vas a servir en otro.** El
artefacto viaja por git, igual que el código. Es la misma lección de la sesión 1, aplicada a
un archivo binario.

In [ ]:
import shutil

paquete = shutil.make_archive(str(RAIZ / "artifacts"), "zip", str(ARTEFACTOS))
print(f"empaquetado: {paquete}")

if EN_COLAB:
    from google.colab import files

    files.download(paquete)
else:
    print("No estas en Colab: el artefacto ya esta en artifacts/, no hay nada que descargar.")

## 15. Qué NO fue al artefacto

Vale la pena decirlo en voz alta:

- **Los datos de entrenamiento.** El artefacto lleva el modelo, no el CSV.
- **Credenciales.** Nunca, en ningún artefacto.
- **Rutas absolutas de esta máquina.** Es el error clásico: un `Pipeline` que congela
  `/content/...` o `/Users/tu-nombre/...` dentro del pickle y falla en el servidor.
  `test_paridad.py` lo comprueba.

Ahora ve a `docs/s2-guia.md` y sigue con el servicio.